In [1]:
# @title 1. Install System Dependencies & Clone ComfyUI (everything lives in /tmp)
import os, subprocess
from IPython.display import clear_output

# ---- Everything heavy goes to /tmp so /kaggle/working stays empty ----
TMP_ROOT  = "/tmp/comfy"
COMFY_DIR = f"{TMP_ROOT}/ComfyUI"

for sub in ("tmp", "hf_cache", "pip_cache", "cache", "torch_cache"):
    os.makedirs(f"{TMP_ROOT}/{sub}", exist_ok=True)

# Redirect every cache / temp location away from /kaggle/working and ~/
os.environ.update({
    "TMPDIR":         f"{TMP_ROOT}/tmp",
    "HF_HOME":        f"{TMP_ROOT}/hf_cache",
    "PIP_CACHE_DIR":  f"{TMP_ROOT}/pip_cache",
    "XDG_CACHE_HOME": f"{TMP_ROOT}/cache",
    "TORCH_HOME":     f"{TMP_ROOT}/torch_cache",
})

# 1. System tools
#    NOTE: 'libgl1-mesa-glx' no longer exists on newer Ubuntu images; it made the whole
#    apt command fail, so aria2 was never installed ("aria2c: not found"). Use 'libgl1'.
!apt-get update -y
!apt-get install -y ffmpeg libgl1 aria2 openssh-client

# 2. Clone ComfyUI into /tmp
if not os.path.exists(COMFY_DIR):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR}
    print("✅ ComfyUI cloned to", COMFY_DIR)
else:
    print("ℹ️ ComfyUI already exists in /tmp.")

# 3. ComfyUI Manager
MANAGER_DIR = f"{COMFY_DIR}/custom_nodes/ComfyUI-Manager"
if not os.path.exists(MANAGER_DIR):
    !git clone --depth 1 https://github.com/ltdrdata/ComfyUI-Manager.git {MANAGER_DIR}
    print("✅ ComfyUI Manager installed.")

# 4. Python requirements (no pip cache -> nothing left behind)
!pip install --no-cache-dir -r {COMFY_DIR}/requirements.txt

clear_output()
print("✅ Installation complete. ComfyUI is in", COMFY_DIR)


✅ Installation complete. ComfyUI is in /tmp/comfy/ComfyUI


In [2]:
# @title 2. Keep /kaggle/working clean & prepare /tmp folders
import os, shutil

# Previous version of this notebook put ComfyUI (and model symlinks) in /kaggle/working.
# Remove that old copy so it stops taking space. Set to False to keep it.
CLEAN_OLD_WORKING_COPY = True

# Where generated images go. False = /tmp (nothing saved in /kaggle/working).
# True = /kaggle/working/comfy_output (handy to download results, but uses working space).
SAVE_OUTPUTS_TO_WORKING = False

OLD_DIR = "/kaggle/working/ComfyUI"
if CLEAN_OLD_WORKING_COPY and os.path.lexists(OLD_DIR):
    if os.path.islink(OLD_DIR):
        os.unlink(OLD_DIR)
    else:
        # unlink symlinks first so rmtree never follows them
        for name in ("models", "input", "output"):
            p = os.path.join(OLD_DIR, name)
            if os.path.islink(p):
                os.unlink(p)
        shutil.rmtree(OLD_DIR, ignore_errors=True)
    print("🧹 Removed old ComfyUI copy from /kaggle/working")

MODELS_DIR = f"{COMFY_DIR}/models"
INPUT_DIR  = f"{TMP_ROOT}/input"
TEMP_DIR   = f"{TMP_ROOT}/comfy_temp"
OUTPUT_DIR = "/kaggle/working/comfy_output" if SAVE_OUTPUTS_TO_WORKING else f"{TMP_ROOT}/output"

for d in (MODELS_DIR, INPUT_DIR, TEMP_DIR, OUTPUT_DIR):
    os.makedirs(d, exist_ok=True)

print("📁 models :", MODELS_DIR)
print("📁 input  :", INPUT_DIR)
print("📁 output :", OUTPUT_DIR)
print("📁 temp   :", TEMP_DIR)
print()
!df -h /tmp /kaggle/working


📁 models : /tmp/comfy/ComfyUI/models
📁 input  : /tmp/comfy/input
📁 output : /tmp/comfy/output
📁 temp   : /tmp/comfy/comfy_temp

Filesystem      Size  Used Avail Use% Mounted on
overlay         8.0T  7.0T  1.1T  87% /
/dev/loop2       20G   80K   20G   1% /kaggle/working


In [3]:
import os
os.environ["HF_TOKEN"] = "hf_your_token_here"  # @param {type:"string"}


In [4]:
import os, shutil, subprocess

BASE = f"{COMFY_DIR}/models"          # lives in /tmp
DIRS = {
    "text_encoders": f"{BASE}/text_encoders",
    "clip_vision": f"{BASE}/clip_vision",
    "loras": f"{BASE}/loras",
    "diffusion_models": f"{BASE}/diffusion_models",
    "vae": f"{BASE}/vae",
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

downloads = [
    # VAE
    ("https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors",
     f"{DIRS['vae']}/wan_2.1_vae.safetensors"),

    # CLIP vision
    ("https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/clip_vision/clip_vision_h.safetensors",
     f"{DIRS['clip_vision']}/clip_vision_h.safetensors"),

    # Text encoder
    ("https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
     f"{DIRS['text_encoders']}/umt5_xxl_fp8_e4m3fn_scaled.safetensors"),

    # LoRA
    ("https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/loras/chronoedit_distill_lora.safetensors",
     f"{DIRS['loras']}/chronoedit_distill_lora.safetensors"),

    # Diffusion model
    ("https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/diffusion_models/chrono_edit_14B_fp16.safetensors",
     f"{DIRS['diffusion_models']}/chrono_edit_14B_fp16.safetensors"),
]

HF_TOKEN = os.environ.get("HF_TOKEN", "")
HAS_TOKEN = bool(HF_TOKEN) and "your_token_here" not in HF_TOKEN

def dl(url, out_path):
    if os.path.exists(out_path) and os.path.getsize(out_path) > 1024 * 1024:
        print("✅ Exists:", os.path.basename(out_path))
        return
    name, folder = os.path.basename(out_path), os.path.dirname(out_path)
    print("⬇️", name)

    if shutil.which("aria2c"):
        cmd = ["aria2c", "-x", "16", "-s", "16", "-c", "-o", name, "-d", folder]
        if HAS_TOKEN:
            cmd += [f"--header=Authorization: Bearer {HF_TOKEN}"]
        cmd += [url]
    else:  # fallback if aria2 failed to install
        print("⚠️ aria2c not found, falling back to wget")
        cmd = ["wget", "-c", "-O", out_path]
        if HAS_TOKEN:
            cmd += [f"--header=Authorization: Bearer {HF_TOKEN}"]
        cmd += [url]
    subprocess.run(cmd, check=True)

print("💾 Free space before download:")
!df -h /tmp

for url, outp in downloads:
    dl(url, outp)

print("\nFolders:")
!ls -lh "{DIRS['text_encoders']}" "{DIRS['clip_vision']}" "{DIRS['loras']}" "{DIRS['diffusion_models']}" "{DIRS['vae']}"
print("\n💾 Free space after download:")
!df -h /tmp /kaggle/working


💾 Free space before download:
Filesystem      Size  Used Avail Use% Mounted on
overlay         8.0T  7.0T  1.1T  87% /
⬇️ wan_2.1_vae.safetensors

10/09 19:55:25 [NOTICE] Downloading 1 item(s)

10/09 19:55:25 [NOTICE] CUID#7 - Redirecting to https://us.gcp.cdn.hf.co/xet-bridge-us/6885cd8c6963bab90aab7f6f/2e358d90f91ce658dc1be053d44cf2d6437e5f517c71ae89fcb2c89fd54ca67e?xip=20Lf-UGH-x4&user_id=public&X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27wan_2.1_vae.safetensors%3B+filename%3D%22wan_2.1_vae.safetensors%22%3B&Expires=1791579325&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjg4NWNkOGM2OTYzYmFiOTBhYWI3ZjZmLzJlMzU4ZDkwZjkxY2U2NThkYzFiZTA1M2Q0NGNmMmQ2NDM3ZTVmNTE3YzcxYWU4OWZjYjJjODlmZDU0Y2E2N2VcXD94aXA9MjBMZi1VR0gteDQmdXNlcl9pZD1wdWJsaWMmWC1YZXQtQ2FzLVVpZD1wdWJsaWMmcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj0qIiwiQ29uZGl0aW9uIjp7IkRhdGVMZXNzVGhhbiI6eyJFcG9jaFRpbWUiOjE3OTE1NzkzMjV9fX1dfQ__&Signature=MEQCIFe

In [8]:
# @title Download MiniMax Music 3 models (into /tmp)
import os, shutil, subprocess

COMFY_DIR = globals().get("COMFY_DIR", "/tmp/comfy/ComfyUI")
BASE = f"{COMFY_DIR}/models"

REPO = "https://huggingface.co/Comfy-Org/MiniMax-Music-3/resolve/main"
downloads = [
    # (url, target folder, filename)
    (f"{REPO}/vae/minimax_music3_dav.safetensors",
     f"{BASE}/vae", "minimax_music3_dav.safetensors"),                                    # ~207 MB
    (f"{REPO}/text_encoders/minimax_music3_text_encoder_pruned_int8_convrot.safetensors",
     f"{BASE}/text_encoders", "minimax_music3_text_encoder_pruned_int8_convrot.safetensors"),  # ~8.6 GB
    (f"{REPO}/diffusion_models/minimax_music3_dit_fp16.safetensors",
     f"{BASE}/diffusion_models", "minimax_music3_dit_fp16.safetensors"),                  # ~4.6 GB
]

HF_TOKEN = os.environ.get("HF_TOKEN", "")
HAS_TOKEN = bool(HF_TOKEN) and "your_token_here" not in HF_TOKEN

def dl(url, folder, name):
    os.makedirs(folder, exist_ok=True)
    out = f"{folder}/{name}"
    if os.path.exists(out) and os.path.getsize(out) > 1024 * 1024:
        print("✅ Exists:", name)
        return
    print("⬇️", name)
    if shutil.which("aria2c"):
        cmd = ["aria2c", "-x", "16", "-s", "16", "-c", "-o", name, "-d", folder]
        if HAS_TOKEN:
            cmd.append(f"--header=Authorization: Bearer {HF_TOKEN}")
        cmd.append(url)
    else:
        cmd = ["wget", "-c", "-O", out]
        if HAS_TOKEN:
            cmd.append(f"--header=Authorization: Bearer {HF_TOKEN}")
        cmd.append(url)
    subprocess.run(cmd, check=True)

print("💾 Free space before:")
!df -h /tmp
for url, folder, name in downloads:
    dl(url, folder, name)
print("\n💾 Free space after:")
!df -h /tmp
!ls -lh {BASE}/vae {BASE}/text_encoders {BASE}/diffusion_models

💾 Free space before:
Filesystem      Size  Used Avail Use% Mounted on
overlay         8.0T  7.0T  1.1T  88% /
⬇️ minimax_music3_dav.safetensors

10/09 20:25:13 [NOTICE] Downloading 1 item(s)

10/09 20:25:13 [NOTICE] CUID#7 - Redirecting to https://us.gcp.cdn.hf.co/xet-bridge-us/6a76a9b12c61823dc5f43615/bedc48b56b6c4e6e9ed038f94d3bf5b3ce10ab0a55a048388f8a60a20b2f7779?X-Xet-Cas-Uid=public&user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27minimax_music3_dav.safetensors%3B+filename%3D%22minimax_music3_dav.safetensors%22%3B&xip=20Lf-UGH-x4&Expires=1791581113&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNmE3NmE5YjEyYzYxODIzZGM1ZjQzNjE1L2JlZGM0OGI1NmI2YzRlNmU5ZWQwMzhmOTRkM2JmNWIzY2UxMGFiMGE1NWEwNDgzODhmOGE2MGEyMGIyZjc3NzlcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZ1c2VyX2lkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomeGlwPTIwTGYtVUdILXg0IiwiQ29uZGl0aW9uIjp7IkRhdGVMZXNzVGhhbiI6eyJFcG9jaFRpbWUiOjE3OTE1ODExMTN9fX1dfQ__&Signa

In [9]:
# @title 3. Run ComfyUI + public tunnel (Cloudflare by default, Pinggy optional)
import subprocess, threading, re, time, os, sys

PORT = 8188
TUNNEL = "cloudflare"   # "cloudflare" (recommended: handles websockets well) or "pinggy"
COMFY_DIR = globals().get("COMFY_DIR", "/tmp/comfy/ComfyUI")   # set in cell 1 (lives in /tmp)
TMP_ROOT  = globals().get("TMP_ROOT", "/tmp/comfy")
INPUT_DIR  = globals().get("INPUT_DIR",  f"{TMP_ROOT}/input")
OUTPUT_DIR = globals().get("OUTPUT_DIR", f"{TMP_ROOT}/output")
TEMP_DIR   = globals().get("TEMP_DIR",   f"{TMP_ROOT}/comfy_temp")

# Make sure ssh exists (Pinggy uses ssh reverse tunneling)
!apt-get update -y && apt-get install -y openssh-client

# Clean old processes
!pkill -f "main.py" || true
!pkill -f "a.pinggy.io" || true
!pkill -f "cloudflared" || true

comfy_proc = None

def run_comfyui():
    global comfy_proc
    cmd = (
        f"{sys.executable} main.py --listen 0.0.0.0 --port {PORT} "
        f"--input-directory {INPUT_DIR} --output-directory {OUTPUT_DIR} "
        f"--temp-directory {TEMP_DIR} "
        f"--enable-cors-header \"*\""
    )
    comfy_proc = subprocess.Popen(cmd, shell=True, cwd=COMFY_DIR)
    comfy_proc.wait()
    print(f"\n❌ ComfyUI process exited with code {comfy_proc.returncode} - see the log above for the error.")


def wait_for_comfyui(timeout=300):
    """Block until ComfyUI answers on localhost, or it dies / times out."""
    import urllib.request
    t0 = time.time()
    while time.time() - t0 < timeout:
        if comfy_proc is not None and comfy_proc.poll() is not None:
            return False
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PORT}/system_stats", timeout=3)
            return True
        except Exception:
            time.sleep(2)
    return False


def run_pinggy():
    cmd = (
        f"ssh -T -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null "
        f"-o ServerAliveInterval=30 -o ServerAliveCountMax=3 "
        f"-o ExitOnForwardFailure=yes "
        f"-p 443 -R0:127.0.0.1:{PORT} a.pinggy.io"
    )

    # stdin=PIPE keeps stdin OPEN. In a notebook stdin is usually closed (EOF), which makes
    # ssh/pinggy end the session immediately and never print a URL.
    p = subprocess.Popen(
        cmd, shell=True,
        stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )

    ansi = re.compile(r"\x1b\[[0-9;?]*[A-Za-z]")
    url_re = re.compile(r"https?://[^\s\"'<>]+")
    printed = False

    for raw in p.stdout:
        line = ansi.sub("", raw).strip()
        if not line:
            continue
        if not printed:
            print("[pinggy]", line)   # raw output, so you can see what Pinggy says
        for url in url_re.findall(line):
            if "dashboard.pinggy.io" in url:
                continue
            if not printed and "pinggy" in url:
                print("\n🚀 ComfyUI Public URL:", url, "\n")
                printed = True
    print("⚠️ Pinggy tunnel process ended (exit code %s). Re-run this cell." % p.poll())


def run_cloudflared():
    import shutil
    exe = f"{TMP_ROOT}/cloudflared"
    if not os.path.exists(exe):
        print("⬇️ Downloading cloudflared...")
        subprocess.run(
            f"curl -L -o {exe} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x {exe}",
            shell=True, check=True)
    p = subprocess.Popen(
        [exe, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
        stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1)
    url_re = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
    printed = False
    for line in p.stdout:
        m = url_re.search(line)
        if m and not printed:
            print("\n🚀 ComfyUI Public URL:", m.group(0), "\n")
            printed = True
        elif not printed and ("ERR" in line or "error" in line.lower()):
            print("[cloudflared]", line.strip())
    print("⚠️ cloudflared ended (exit code %s). Re-run this cell." % p.poll())


t1 = threading.Thread(target=run_comfyui, daemon=True)
t2 = threading.Thread(target=run_cloudflared if TUNNEL == "cloudflare" else run_pinggy, daemon=True)
t1.start()

print("⏳ Waiting for ComfyUI to come up on localhost (can take a minute)...")
if wait_for_comfyui():
    print(f"✅ ComfyUI is answering on http://127.0.0.1:{PORT}")
    t2.start()
    print(f"⏳ Waiting for the {TUNNEL} tunnel to print the public URL...")
else:
    print("❌ ComfyUI never became reachable on localhost, so no tunnel was started. "
          "Check the log above for the error and paste it here.")


Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease     
Hit:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease                     
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease                         
Hit:6 http://security.ubuntu.com/ubuntu noble-security InRelease               
Hit:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease                 
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Reading package lists... Done
W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8

[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[INFO] Setting output directory to: /tmp/comfy/output
[INFO] Setting input directory to: /tmp/comfy/input
[INFO] [ComfyUI-Manager] Using `uv` as Python module for pip operations.
Using Python 3.13.15 environment at: /usr


[START] Security scan
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-10-09 20:31:38.738
** Platform: Linux
** Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /tmp/comfy/ComfyUI
** ComfyUI Base Folder Path: /tmp/comfy/ComfyUI
** User directory: /tmp/comfy/ComfyUI/user
** ComfyUI-Manager config path: /tmp/comfy/ComfyUI/user/__manager/config.ini
** Log path: /tmp/comfy/ComfyUI/user/comfyui.log


Using Python 3.13.15 environment at: /usr
Using Python 3.13.15 environment at: /usr
[INFO] 
Prestartup times for custom nodes:
[INFO]    0.9 seconds: /tmp/comfy/ComfyUI/custom_nodes/ComfyUI-Manager
[INFO] 
[WARNING] WARNING: You need pytorch with cu130 or higher to use optimized CUDA operations.
WARNING WARNING WARNING
If you are on nvidia 20 series and above it is required that you update your pytorch to cu130 or higher.

[INFO] Found comfy_kitchen backend ascend: {'available': False, 'disabled': False, 'unavailable_reason': "torch-npu is not installed: No module named 'torch_npu'", 'capabilities': []}
[INFO] Found comfy_kitchen backend cuda: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', 'apply_rope1_', 'apply_rope_', 'apply_rope_split_half', 'apply_rope_split_half1', 'apply_rope_split_half1_', 'apply_rope_split_half_', 'convrot_w4a4_linear', 'dequantize_convrot_w4a4_weight', 'dequantize_int8_convrot_weight', '

✅ ComfyUI is answering on http://127.0.0.1:8188
⏳ Waiting for the cloudflare tunnel to print the public URL...

🚀 ComfyUI Public URL: https://sons-occurred-customized-reservoir.trycloudflare.com 



[WARNING] [DEPRECATION WARNING] Detected import of deprecated legacy API: /scripts/ui.js. This is likely caused by a custom node extension using outdated APIs. Please update your extensions or contact the extension author for an updated version.
[WARNING] [DEPRECATION WARNING] Detected import of deprecated legacy API: /extensions/core/groupNode.js. This is likely caused by a custom node extension using outdated APIs. Please update your extensions or contact the extension author for an updated version.
[INFO] Seeder start (roots=('models', 'input', 'output'), phase=full)
[INFO] [assets-event] seeder.scan_started phase=full
[INFO] Asset scan [models] directory: /tmp/comfy/ComfyUI/models
[INFO] Asset scan [input] directories: ['/tmp/comfy/input']
[INFO] Asset scan [output] directories: ['/tmp/comfy/output']
[INFO] Fast scan complete: 0.031s total (created=0, skipped=8, total_paths=8)
[INFO] Scan(('models', 'input', 'output'), full) done 0.040s: created=0 enriched=0 skipped=8
[INFO] [asset

Cannot connect to comfyregistry.
FETCH DATA from: https://raw.githubusercontent.com/ltdrdata/ComfyUI-Manager/main/custom-node-list.json [DONE]


[INFO] got prompt
[INFO] Model storage policy: fast_disk=False paths=['/tmp/comfy/ComfyUI/models/vae/minimax_music3_dav.safetensors']
[INFO] VAE load device: cuda:0, offload device: cpu, dtype: torch.float32
[INFO] Found quantization metadata version 1
[INFO] Model storage policy: fast_disk=False paths=['/tmp/comfy/ComfyUI/models/text_encoders/minimax_music3_text_encoder_pruned_int8_convrot.safetensors']
[INFO] CLIP/text encoder model load device: cuda:0, offload device: cpu, current: cpu, dtype: torch.float16
[INFO] Requested to load MiniMaxMusic3TEModel
[INFO] Model MiniMaxMusic3TEModel prepared for dynamic VRAM loading. 8758MB Staged. 0 patches attached. Force pre-loaded 154 weights: 674 KB.
AR sampling: 100%|██████████| 1501/1501 [28:19<00:00,  1.13s/it]                                  
[INFO] Comfy model compiler graph breaks: 1500, rogues: 0
[INFO] model weight dtype torch.float16, manual cast: None
[INFO] model_type FLOW
[INFO] Model storage policy: fast_disk=False paths=['/tmp

In [6]:
# @title 4. Diagnose: is it ComfyUI, the tunnel, or the browser?
PUBLIC_URL = "https://subsequently-desktops-automotive-administrators.trycloudflare.com"  # paste your current link

import subprocess
def sh(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[:1500], "\n")

print("=== LOCAL (inside Kaggle) ===")
sh("curl -s -o /dev/null -w 'status=%{http_code}\\n' http://127.0.0.1:8188/")
sh("curl -s http://127.0.0.1:8188/system_stats | head -c 600")
print("=== PUBLIC (through the tunnel) ===")
sh(f"curl -s -o /dev/null -w 'status=%{{http_code}}\\n' {PUBLIC_URL}/")
sh(f"curl -s {PUBLIC_URL}/system_stats | head -c 600")
sh(f"curl -s -I {PUBLIC_URL}/ | head -20")
print("=== ComfyUI log (last lines) ===")
sh(f"tail -n 25 {COMFY_DIR}/user/comfyui.log")


=== LOCAL (inside Kaggle) ===
$ curl -s -o /dev/null -w 'status=%{http_code}\n' http://127.0.0.1:8188/
status=200 

$ curl -s http://127.0.0.1:8188/system_stats | head -c 600
{"system": {"os": "linux", "ram_total": 32212254720, "ram_free": 29250183168, "comfyui_version": "0.39.0", "required_frontend_version": "1.55.14", "installed_templates_version": "0.11.78", "required_templates_version": "0.11.78", "comfy_package_versions": [{"name": "comfyui-frontend-package", "installed": "1.55.14", "required": "1.55.14"}, {"name": "comfyui-workflow-templates", "installed": "0.11.78", "required": "0.11.78"}, {"name": "comfyui-embedded-docs", "installed": "0.5.13", "required": "0.5.13"}, {"name": "comfy-kitchen", "installed": "0.2.37", "required": "0.2.37"}, {"name": "comfy-aim 

=== PUBLIC (through the tunnel) ===
$ curl -s -o /dev/null -w 'status=%{http_code}\n' https://subsequently-desktops-automotive-administrators.trycloudflare.com/
status=000 

$ curl -s https://subsequently-desktops-automoti

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
